# Systematics: GENIE

GENIE systematics are handled differently for uncertainty on the event rates vs. uncertainty on the xsec measurement

For the latter, we only consider the impact on response for the signal channel

# GENIE uncertainties

- impact on signal vs. background

- impact on rate, vs. efficiency vs. smearing


In [ ]:
import pandas as pd
import numpy as np
import sys
from os import path, makedirs
from datetime import datetime

# local imports
# sys.path.append('../../../')
sys.path.append('/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.categories import *
from makedf.geniesyst import *
from analysis_village.numucc_1p0pi.utils import *

import matplotlib.pyplot as plt 
from matplotlib.patches import Patch

plt.style.use("presentation.mplstyle")

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

import pickle
from analysis_village.numucc_1p0pi.utils import *
from pyanalib.covariance import *
from pyanalib.variable_calculator import *

In [ ]:
var_configs = [
    VariableConfig.all_events(),
    # VariableConfig.vertex_x(),
    # VariableConfig.vertex_y(),
    # VariableConfig.vertex_z(),
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    # VariableConfig.muon_direction_x(),
    # VariableConfig.muon_direction_y(),
    VariableConfig.proton_momentum(),
    VariableConfig.proton_direction(),
    # VariableConfig.proton_direction_x(),
    # VariableConfig.proton_direction_y(),
    # VariableConfig.opening_angle(),
    VariableConfig.tki_del_Tp(),
    VariableConfig.tki_del_Tp_x(),
    VariableConfig.tki_del_Tp_y(),
    VariableConfig.tki_del_p(),
    VariableConfig.tki_del_alpha(),
    VariableConfig.tki_del_phi(),
    ]

In [ ]:
save_fig = False
show_plot = True

save_fig_base_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi"
today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics_studies_genie-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

In [ ]:
cov_mat_dict = {}
for var_config in var_configs:
    cov_mat_dict[var_config.var_save_name] = {
        # Ar23+ (all)
        "genie": np.zeros((len(var_config.bin_centers), len(var_config.bin_centers))),
        "genie_rate": np.zeros((len(var_config.bin_centers), len(var_config.bin_centers))),
        # Ar23
        "genie_ar23": np.zeros((len(var_config.bin_centers), len(var_config.bin_centers))),
        "genie_ar23_rate": np.zeros((len(var_config.bin_centers), len(var_config.bin_centers))),
    }

In [ ]:
# load per-knob cov matrices from precalculated files

base_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi"
date_str = "20260216"

# Ar23
for genie_tag in ["CCQE", "MEC", "RES", "nonRES", "DIS", "Other"]:
    unc_file = path.join(base_dir, f"systematics-genie-{genie_tag}-{date_str}", f"genie-{genie_tag}_syst_dict.npz")
    unc_arr = np.load(unc_file, allow_pickle=True)
    unc_dict = dict(unc_arr)

    for var_config in var_configs:
        syst_keys = list(unc_dict[var_config.var_save_name].item().keys())
        for k in syst_keys:
            syst_uncert_xsec = np.sqrt(np.diag(unc_dict[var_config.var_save_name].item()[k]["xsec"]["cov_frac"]))
            syst_uncert_rate = np.sqrt(np.diag(unc_dict[var_config.var_save_name].item()[k]["rate"]["cov_frac"]))

            cov_mat_dict[var_config.var_save_name][k] = unc_dict[var_config.var_save_name].item()[k]["xsec"]["cov_frac"]
            cov_mat_dict[var_config.var_save_name][k+"_rate"] = unc_dict[var_config.var_save_name].item()[k]["rate"]["cov_frac"]

            cov_mat_dict[var_config.var_save_name]["genie"] += unc_dict[var_config.var_save_name].item()[k]["xsec"]["cov_frac"]
            cov_mat_dict[var_config.var_save_name]["genie_rate"] += unc_dict[var_config.var_save_name].item()[k]["rate"]["cov_frac"]

            cov_mat_dict[var_config.var_save_name]["genie_ar23"] += unc_dict[var_config.var_save_name].item()[k]["xsec"]["cov_frac"]
            cov_mat_dict[var_config.var_save_name]["genie_ar23_rate"] += unc_dict[var_config.var_save_name].item()[k]["rate"]["cov_frac"]


genie_tag = "Ar23p"
unc_file = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/genie-Ar23p_syst_dict.npz"
unc_arr = np.load(unc_file, allow_pickle=True)
unc_dict = dict(unc_arr)

for k in ar23p_genie_systematics:
    var_keys = list(unc_dict[k].item().keys())
    for var_key in [var_config.var_save_name for var_config in var_configs]:
        syst_uncert_xsec = np.sqrt(np.diag(unc_dict[k].item()[var_key]["xsec"]["cov_frac"]))
        syst_uncert_rate = np.sqrt(np.diag(unc_dict[k].item()[var_key]["rate"]["cov_frac"]))

        cov_mat_dict[var_key][k] = unc_dict[k].item()[var_key]["xsec"]["cov_frac"]
        cov_mat_dict[var_key][k+"_rate"] = unc_dict[k].item()[var_key]["rate"]["cov_frac"]

        cov_mat_dict[var_key]["genie"] += unc_dict[k].item()[var_key]["xsec"]["cov_frac"]
        cov_mat_dict[var_key]["genie_rate"] += unc_dict[k].item()[var_key]["rate"]["cov_frac"]

In [ ]:
ar23p_cov_mat_dict = {}
for var_config in var_configs:
    ar23p_cov_mat_dict[var_config.var_save_name] = {
    }

ar23p_genie_systematics = [
    'ZExpPCAWeighter_SBNNuSyst_multisigma_D_ZExp_b1',
    'ZExpPCAWeighter_SBNNuSyst_multisigma_D_ZExp_b2',
    'ZExpPCAWeighter_SBNNuSyst_multisigma_D_ZExp_b3',
    'ZExpPCAWeighter_SBNNuSyst_multisigma_D_ZExp_b4',
    'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b1',
    'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b2',
    'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b3',
    'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b4',
    'CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin1',
    'CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin2',
    'CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin3',
    'CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin4',
    'CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin5',
    'CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin1',
    'CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin2',
    'CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin3',
    'CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin4',
    'CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin5',
    'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_0',
    'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_1',
    'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_2',
    'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_3',
    'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_4',
    'QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_5',
    'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_VecFFCCQEshape',
    'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_CoulombCCQE',
    'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_NormCCMEC',
    'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_NormNCMEC',
    'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_DecayAngMEC',
    'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_MFP_pi',
    'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrCEx_pi',
    'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrInel_pi',
    'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrAbs_pi',
    'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrPiProd_pi',
    'MECq0q3InterpWeighting_SuSAv2ToValenica_q0binned_MECResponse_q0bin0',
    'MECq0q3InterpWeighting_SuSAv2ToValenica_q0binned_MECResponse_q0bin1',
    'MECq0q3InterpWeighting_SuSAv2ToValenica_q0binned_MECResponse_q0bin2',
    'MECq0q3InterpWeighting_SuSAv2ToValenica_q0binned_MECResponse_q0bin3',
    'MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse_q0bin0',
    'MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse_q0bin1',
    'MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse_q0bin2',
    'MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse_q0bin3'
]

# Ar23+ has grouped syst knobs
# sum matrices for each group
groups = [
            # 'ZExpPCAWeighter_SBNNuSyst_multisigma_D_ZExp',
            # 'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp',
            "CCQETemplateReweight_SBNNuSyst_multisigma_SF", "CCQETemplateReweight_SBNNuSyst_multisigma_CRPA",
            "QEInterference_SBNNuSyst_multisigma_INT_QEIntf", "MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse",
            # 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_VecFFCCQEshape',
            # 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_CoulombCCQE',
            # 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_NormCCMEC',
            # 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_NormNCMEC',
            # 'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_DecayAngMEC',
            'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_MFP_pi',
            'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrCEx_pi',
            'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrInel_pi',
            'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrAbs_pi',
            'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_FrPiProd_pi',
]

genie_tag = "Ar23p"
# unc_file = path.join(base_dir, f"systematics-genie-{genie_tag}-{date_str}", f"genie-{genie_tag}_syst_dict.npz")
unc_file = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/genie-Ar23p_syst_dict.npz"
unc_arr = np.load(unc_file, allow_pickle=True)
unc_dict = dict(unc_arr)

for var_config in var_configs:
    for group in groups:
        group_syst_names = [k for k in ar23p_genie_systematics if group in k]
        ar23p_cov_mat_dict[var_config.var_save_name][group] = np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))
        ar23p_cov_mat_dict[var_config.var_save_name][group+"_rate"] = np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))

        for k in group_syst_names:
            print(k)
            syst_uncert_xsec = np.sqrt(np.diag(unc_dict[k].item()[var_config.var_save_name]["xsec"]["cov_frac"]))
            syst_uncert_rate = np.sqrt(np.diag(unc_dict[k].item()[var_config.var_save_name]["rate"]["cov_frac"]))

            ar23p_cov_mat_dict[var_config.var_save_name][group] += unc_dict[k].item()[var_config.var_save_name]["xsec"]["cov_frac"]
            ar23p_cov_mat_dict[var_config.var_save_name][group+"_rate"] += unc_dict[k].item()[var_config.var_save_name]["rate"]["cov_frac"]

groups = [
            'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp',
]

unc_file = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/genie-Ar23p_syst_dict.npz"
unc_arr = np.load(unc_file, allow_pickle=True)
unc_dict = dict(unc_arr)

for var_config in var_configs:
    for group in groups:
        group_syst_names = [k for k in ar23p_genie_systematics if group in k]
        ar23p_cov_mat_dict[var_config.var_save_name][group] = np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))
        ar23p_cov_mat_dict[var_config.var_save_name][group+"_rate"] = np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))

        for k in group_syst_names:
            print(k)
            syst_uncert_xsec = np.sqrt(np.diag(unc_dict[k].item()[var_config.var_save_name]["xsec"]["cov_frac"]))
            syst_uncert_rate = np.sqrt(np.diag(unc_dict[k].item()[var_config.var_save_name]["rate"]["cov_frac"]))

            ar23p_cov_mat_dict[var_config.var_save_name][group] += unc_dict[k].item()[var_config.var_save_name]["xsec"]["cov_frac"]
            ar23p_cov_mat_dict[var_config.var_save_name][group+"_rate"] += unc_dict[k].item()[var_config.var_save_name]["rate"]["cov_frac"]

CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin1
CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin2
CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin3
CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin4
CCQETemplateReweight_SBNNuSyst_multisigma_SF_q0bin5
CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin1
CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin2
CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin3
CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin4
CCQETemplateReweight_SBNNuSyst_multisigma_CRPA_q0bin5
QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_0
QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_1
QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_2
QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_3
QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_4
QEInterference_SBNNuSyst_multisigma_INT_QEIntf_dial_5
MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse_q0bin0
MECq0q3InterpWeighting_SuSAv2ToMartini_q0binned_MECResponse_q0bin1
MECq0q3Inter

In [ ]:
ar23p_cov_mat_dict[var_config.var_save_name]

{'CCQETemplateReweight_SBNNuSyst_multisigma_SF': array([[ 2.24917355e-05,  3.57590770e-05,  2.90313548e-05,
          4.41172966e-05,  8.01705264e-06,  3.15321103e-06,
         -2.12580684e-05, -1.85601542e-05, -3.81233746e-05,
         -5.33977296e-05, -8.90364007e-05],
        [ 3.57590770e-05,  9.06528068e-05,  5.23082900e-05,
          6.88084613e-05,  5.37659819e-06, -2.29719147e-05,
         -1.35552933e-04, -7.38681053e-05, -1.86661444e-04,
         -2.58144378e-04, -3.04161606e-04],
        [ 2.90313548e-05,  5.23082900e-05,  4.88706185e-05,
          7.72672938e-05,  1.99503854e-05,  1.89094599e-07,
         -4.24109142e-05, -4.04981841e-05, -7.54272985e-05,
         -1.06973059e-04, -1.62267665e-04],
        [ 4.41172966e-05,  6.88084613e-05,  7.72672938e-05,
          1.27875715e-04,  3.80860540e-05,  1.00136425e-05,
         -2.96964780e-05, -5.11301312e-05, -7.57132308e-05,
         -1.09113882e-04, -2.02448927e-04],
        [ 8.01705264e-06,  5.37659819e-06,  1.99503854e-

In [ ]:
group = "FSI"
genie_tag = "Other"
unc_file = path.join(base_dir, f"systematics-genie-{genie_tag}-{date_str}", f"genie-{genie_tag}_syst_dict.npz")
unc_arr = np.load(unc_file, allow_pickle=True)
unc_dict = dict(unc_arr)

for var_config in var_configs:
    group_syst_names = other_genie_systematics
    cov_mat_dict[var_config.var_save_name][group] = np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))
    cov_mat_dict[var_config.var_save_name][group+"_rate"] = np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))

    for k in group_syst_names:
        syst_uncert_xsec = np.sqrt(np.diag(unc_dict[var_config.var_save_name].item()[k]["xsec"]["cov_frac"]))
        syst_uncert_rate = np.sqrt(np.diag(unc_dict[var_config.var_save_name].item()[k]["rate"]["cov_frac"]))

        cov_mat_dict[var_config.var_save_name][group] += unc_dict[var_config.var_save_name].item()[k]["xsec"]["cov_frac"]
        cov_mat_dict[var_config.var_save_name][group+"_rate"] += unc_dict[var_config.var_save_name].item()[k]["rate"]["cov_frac"]

In [ ]:
syst_names = ar23p_genie_systematics
systs = ar23p_genie_systematics
for var_config in var_configs:
    save_fig_name = "{}/genie_breakdown-{}-rate.pdf".format(save_fig_dir, var_config.var_save_name)
    plot_syst_uncert(cov_mat_dict, var_config, syst_names, systs, syst_on="rate", save_fig=True, save_fig_name=save_fig_name)
    save_fig_name = "{}/genie_breakdown-{}-xsec.pdf".format(save_fig_dir, var_config.var_save_name)
    plot_syst_uncert(cov_mat_dict, var_config, syst_names, systs, syst_on="xsec", save_fig=True, save_fig_name=save_fig_name)

NameError: name 'plot_syst_uncert' is not defined

In [ ]:
# compare total uncertainty between Ar23 vs. Ar23+

var_config = VariableConfig.tki_del_Tp()

group = "genie_ar23"
ar23_unc = np.sqrt(np.diag(cov_mat_dict[var_config.var_save_name][group]))
ar23_unc_rate = np.sqrt(np.diag(cov_mat_dict[var_config.var_save_name][group+"_rate"]))
print("Ar23 Total uncertainty on signal rate: ", ar23_unc_rate)
print("Ar23 Total uncertainty on signal cross-section measurement: ", ar23_unc)

group = "genie"
ar23p_unc = np.sqrt(np.diag(cov_mat_dict[var_config.var_save_name][group]))
ar23p_unc_rate = np.sqrt(np.diag(cov_mat_dict[var_config.var_save_name][group+"_rate"]))
print("Ar23+ Total uncertainty on signal rate: ", ar23p_unc_rate)
print("Ar23+ Total uncertainty on signal cross-section measurement: ", ar23p_unc)
ar23_unc[-1] = 0.24

unc_list = [ar23_unc, ar23p_unc]
unc_rate_list = [ar23_unc_rate, ar23p_unc_rate]
legends = ["Ar23", "Ar23+"]
plot_labels = ["", "", "Total Uncertainty on Signal Rate"]
save_name = save_fig_dir + f"/{var_config.var_save_name}-ar23_vs_ar23p-total_uncert_rate.pdf"
plot_frac_unc(unc_rate_list, var_config, legends=legends, plot_labels=plot_labels, save_fig=save_fig, save_name=save_name)
plot_labels = ["", "", "Total Uncertainty on Signal Cross-Section Measurement"]
save_name = save_fig_dir + f"/{var_config.var_save_name}-ar23_vs_ar23p-total_uncert_xsec.pdf"
plot_frac_unc(unc_list, var_config, legends=legends, plot_labels=plot_labels, save_fig=save_fig, save_name=save_name)

In [ ]:
# # save cov_mat_dict
# import pickle
# pickle.dump(cov_mat_dict, open(f"/exp/sbnd/data/users/munjung/plots/numucc1p0pi/cov_mat_dict-{date_str}.pkl", "wb"))
# print(date_str)

In [ ]:
# Decompose fractional uncertainties (sqrt of diagonal) into PCA components and plot them
from sklearn.decomposition import PCA

def get_pca_explained(cov, n_components=None):
    # PCA expects features as columns, so transpose if needed
    # Covariance matrix is symmetric, so eigendecomposition is sufficient
    eigvals, eigvecs = np.linalg.eigh(cov)
    # Sort by descending eigenvalue
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    # Explained variance ratios
    explained_var = eigvals / np.sum(eigvals)
    return eigvals, eigvecs, explained_var


var_config = VariableConfig.tki_del_Tp()

group = "genie_ar23"
ar23_cov = cov_mat_dict[var_config.var_save_name][group]
ar23_cov_rate = cov_mat_dict[var_config.var_save_name][group+"_rate"]

group = "genie"
ar23p_cov = cov_mat_dict[var_config.var_save_name][group]
ar23p_cov_rate = cov_mat_dict[var_config.var_save_name][group+"_rate"]

ar23_eigvals, ar23_eigvecs, ar23_explained = get_pca_explained(ar23_cov)
ar23_eigvals_rate, ar23_eigvecs_rate, ar23_explained_rate = get_pca_explained(ar23_cov_rate)
ar23p_eigvals, ar23p_eigvecs, ar23p_explained = get_pca_explained(ar23p_cov)
ar23p_eigvals_rate, ar23p_eigvecs_rate, ar23p_explained_rate = get_pca_explained(ar23p_cov_rate)

plt.figure()
plt.bar(range(1, len(ar23_explained) + 1), ar23_explained, alpha=0.7, label='Ar23')
plt.bar(range(1, len(ar23p_explained) + 1), ar23p_explained, alpha=0.7, label='Ar23+')
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.legend()
plt.title("PCA Explained Variance (Cross-Section Covariance)")
plt.show()

In [ ]:
# compare FSI uncertainty between Ar23 vs. Ar23+
var_config = VariableConfig.all_events()

group = "FSI"
ar23_unc = np.sqrt(np.diag(cov_mat_dict[var_config.var_save_name][group]))
ar23_unc_rate = np.sqrt(np.diag(cov_mat_dict[var_config.var_save_name][group+"_rate"]))
print("Ar23 FSI uncertainty on signal rate: ", ar23_unc_rate)
print("Ar23 FSI uncertainty on signal cross-section measurement: ", ar23_unc)

group = "GENIEReWeight_SBNNuSyst_multisigma_EDepFSI"
ar23p_unc = np.sqrt(np.diag(cov_mat_dict[var_config.var_save_name][group]))
ar23p_unc_rate = np.sqrt(np.diag(cov_mat_dict[var_config.var_save_name][group+"_rate"]))
print("Ar23+ FSI uncertainty on signal rate: ", ar23p_unc_rate)
print("Ar23+ FSI uncertainty on signal cross-section measurement: ", ar23p_unc)

unc_list = [ar23_unc, ar23p_unc]
unc_rate_list = [ar23_unc_rate, ar23p_unc_rate]
legends = ["Ar23", "Ar23+"]
plot_labels = ["", "", "FSI Uncertainty on Signal Rate"]
save_name = save_fig_dir + f"/{var_config.var_save_name}-ar23_vs_ar23p-FSI_uncert_rate.pdf"
plot_frac_unc(unc_rate_list, var_config, legends=legends, plot_labels=plot_labels, save_fig=save_fig, save_name=save_name)
plot_labels = ["", "", "FSI Uncertainty on Signal Cross-Section Measurement"]
save_name = save_fig_dir + f"/{var_config.var_save_name}-ar23_vs_ar23p-FSI_uncert_xsec.pdf"
plot_frac_unc(unc_list, var_config, legends=legends, plot_labels=plot_labels, save_fig=save_fig, save_name=save_name)

In [ ]:
var_config = VariableConfig.tki_del_Tp()

unc_list = []
unc_rate_list = []
group = groups[7]
print(group)
group_syst_names = [k for k in ar23p_genie_systematics if group in k]
for k in group_syst_names:
    this_unc = np.sqrt(np.diag(unc_dict[var_config.var_save_name].item()[k]["xsec"]["cov_frac"]))
    this_unc_rate = np.sqrt(np.diag(unc_dict[var_config.var_save_name].item()[k]["rate"]["cov_frac"]))
    unc_list.append(this_unc)
    unc_rate_list.append(this_unc_rate)

total_unc = np.sqrt(np.diag(cov_mat_dict[var_config.var_save_name][group]))
total_unc_rate = np.sqrt(np.diag(cov_mat_dict[var_config.var_save_name][group+"_rate"]))

plot_frac_unc(unc_list, var_config)
# plot_frac_unc(unc_rate_list, var_config)

plot_frac_unc([total_unc], var_config)
# plot_frac_unc([total_unc_rate], var_config)


In [ ]:
var_config = VariableConfig.tki_del_Tp()

genie_cov = cov_mat_dict[var_config.var_save_name]["genie"]
plt.imshow(genie_cov)
plt.colorbar()
plt.show();

plot_frac_unc([np.sqrt(np.diag(cov_mat_dict[var_config.var_save_name]["genie"]))], var_config)
